In [0]:
%sql
USE CATALOG ecommerce;
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
catalogo      = "ecommerce"
bronze_schema = "bronze"
silver_schema = "silver"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_customers")

# Mantém apenas o registro mais recente por consumidor,
# evitando duplicatas causadas por múltiplos pedidos ou reingestões
window_dedup = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

df_consumidores = (df
    # Adiciona coluna de ranking para deduplicação
    .withColumn("rank", F.row_number().over(window_dedup))
    # Mantém apenas o registro mais recente (rank = 1)
    .filter(F.col("rank") == 1)
    # Tradução das colunas
    .withColumnRenamed("customer_id",              "id_consumidor")
    .withColumnRenamed("customer_zip_code_prefix", "prefixo_cep")
    .withColumnRenamed("customer_unique_id",       "id_consumidor_unico")
    .withColumnRenamed("customer_city",            "cidade")
    .withColumnRenamed("customer_state",           "estado")
    .withColumnRenamed("customer_name",            "nome_consumidor")
    .withColumnRenamed("customer_gender",            "genero_consumidor")
    .withColumnRenamed("customer_bithdate",            "data_nascimento")
    .withColumnRenamed("customer_age",            "idade_consumidor")
    # Padronização em maiúsculas para evitar inconsistências em agrupamentos
    .withColumn("cidade", F.upper(F.col("cidade")))
    .withColumn("estado", F.upper(F.col("estado")))
    .drop("rank", "timestamp_ingestion")
)

# Apenas para validar
#df_consumidores.display()

df_consumidores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_schema}.dim_consumidores")

print(f"✓ dim_consumidores ({df_consumidores.count()} linhas)")

✓ dim_consumidores (99441 linhas)


In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_orders")

# Tradução do status para português para uso em relatórios de negócio
status_expr = F.col("order_status")
for en, pt in {
    "delivered":   "entregue",
    "canceled":    "cancelado",
    "shipped":     "enviado",
    "processing":  "processando",
    "invoiced":    "faturado",
    "unavailable": "indisponível",
    "created":     "criado",
    "approved":    "aprovado",
}.items():
    status_expr = F.when(F.col("order_status") == en, pt).otherwise(status_expr)

df_pedidos = (df
    .withColumnRenamed("order_id",    "id_pedido")
    .withColumnRenamed("customer_id", "id_consumidor")
    .withColumn("status", status_expr)
    .withColumn("data_pedido",           F.to_timestamp("order_purchase_timestamp"))
    .withColumn("data_aprovacao",        F.to_timestamp("order_approved_at"))
    .withColumn("data_entrega_real",     F.to_timestamp("order_delivered_customer_date"))
    .withColumn("data_entrega_estimada", F.to_timestamp("order_estimated_delivery_date"))
    # Métricas de SLA logístico: avaliam se os prazos prometidos foram cumpridos
    .withColumn("tempo_entrega_dias",
        F.datediff("data_entrega_real", "data_pedido"))
    .withColumn("tempo_entrega_estimado_dias",
        F.datediff("data_entrega_estimada", "data_pedido"))
    .withColumn("diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))
    # Negativo ou zero = dentro do prazo; positivo = atrasado
    .withColumn("entrega_no_prazo",
        F.when(F.col("status") != "entregue", "Não Entregue")
         .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
         .otherwise("Não"))
    .drop("order_status", "order_purchase_timestamp", "order_approved_at",
          "order_delivered_carrier_date", "order_delivered_customer_date",
          "order_estimated_delivery_date", "timestamp_ingestion")
)

# Apenas para validar
#df_pedidos.display()

df_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_schema}.fat_pedidos")

print(f"✓ fat_pedidos ({df_pedidos.count()} linhas)")

✓ fat_pedidos (99441 linhas)


In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_order_items")

# Granularidade mais fina das vendas: um item por linha dentro de cada pedido
df_itens = (df
    .withColumnRenamed("order_id",      "id_pedido")
    .withColumnRenamed("order_item_id", "id_item")
    .withColumnRenamed("product_id",    "id_produto")
    .withColumnRenamed("seller_id",     "id_vendedor")
    .withColumnRenamed("price",         "preco_BRL")
    .withColumnRenamed("freight_value", "preco_frete")
    .drop("shipping_limit_date", "timestamp_ingestion")
)

# Apenas para validar
#df_itens.display()

(df_itens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_schema}.fat_itens_pedidos"))

print(f"✓ fat_itens_pedidos ({df_itens.count()} linhas)")

✓ fat_itens_pedidos (112650 linhas)


In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_order_payments")

# Tradução do tipo de pagamento para segmentação de receita em português
df_pagamentos = (df
    .withColumnRenamed("order_id",             "id_pedido")
    .withColumnRenamed("payment_sequential",   "sequencial_pagamento")
    .withColumnRenamed("payment_installments", "parcelas")
    .withColumnRenamed("payment_value",        "valor_pagamento")
    .withColumn("tipo_pagamento",
        F.when(F.col("payment_type") == "credit_card", "Cartão de Crédito")
         .when(F.col("payment_type") == "boleto",      "Boleto")
         .when(F.col("payment_type") == "voucher",     "Voucher")
         .when(F.col("payment_type") == "debit_card",  "Cartão de Débito")
         .otherwise("Não Definido"))
    .drop("payment_type", "timestamp_ingestion")
)

# Apenas para validar
#df_pagamentos.display()

(df_pagamentos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_schema}.fat_pagamentos_pedidos"))

print(f"✓ fat_pagamentos_pedidos ({df_pagamentos.count()} linhas)")

✓ fat_pagamentos_pedidos (103886 linhas)


In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_order_reviews")

df_avaliacoes = (df
    .withColumnRenamed("review_id",              "id_avaliacao")
    .withColumnRenamed("order_id",               "id_pedido")
    .withColumnRenamed("review_score",           "nota_avaliacao")
    .withColumnRenamed("review_comment_title",   "titulo_avaliacao")
    .withColumnRenamed("review_comment_message", "comentario_avaliacao")
    # try_to_timestamp evita falhas no pipeline por datas malformadas,
    # retornando null em vez de quebrar o processamento
    .withColumn("data_criacao_avaliacao",
        F.try_to_timestamp("review_creation_date"))
    .withColumn("data_resposta_avaliacao",
        F.try_to_timestamp("review_answer_timestamp"))
    # Remove avaliações sem pedido associado — não têm valor analítico
    .filter(F.col("id_pedido").isNotNull())
    # Remove datas no futuro — indicam dados corrompidos ou de teste
    .filter(
        F.col("data_criacao_avaliacao").isNull() |
        (F.col("data_criacao_avaliacao") <= F.current_timestamp()))
    # Preenche textos vazios para manter consistência nos relatórios
    .withColumn("titulo_avaliacao",
        F.when(
            F.col("titulo_avaliacao").isNull() | (F.col("titulo_avaliacao") == ""),
            "Sem título")
         .otherwise(F.col("titulo_avaliacao")))
    .withColumn("comentario_avaliacao",
        F.when(
            F.col("comentario_avaliacao").isNull() | (F.col("comentario_avaliacao") == ""),
            "Sem comentário")
         .otherwise(F.col("comentario_avaliacao")))
    .drop("review_creation_date", "review_answer_timestamp", "timestamp_ingestion")
)

# Apenas para validar
#df_avaliacoes.display()

(df_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_schema}.fat_avaliacoes_pedidos"))

print(f"✓ fat_avaliacoes_pedidos ({df_avaliacoes.count()} linhas)")

✓ fat_avaliacoes_pedidos (101926 linhas)


In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_products")

# Deduplicação: mantém apenas o registro mais recente por produto
window_dedup = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

df_produtos = (df
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .withColumnRenamed("product_id",                 "id_produto")
    .withColumnRenamed("product_name",               "nome_produto")
    .withColumnRenamed("product_category_name",      "categoria_produto")
    .withColumnRenamed("product_name_lenght",        "tamanho_nome_produto")
    .withColumnRenamed("product_description_lenght", "tamanho_descricao_produto")
    .withColumnRenamed("product_photos_qty",         "quantidade_fotos")
    .withColumnRenamed("product_weight_g",           "peso_produto_gramas")
    .withColumnRenamed("product_length_cm",          "comprimento_centimetros")
    .withColumnRenamed("product_height_cm",          "altura_centimetros")
    .withColumnRenamed("product_width_cm",           "largura_centimetros")
    .drop("rank", "timestamp_ingestion")
)

# Apenas para validar
#df_produtos.display()

(df_produtos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_schema}.dim_produtos"))

print(f"✓ dim_produtos ({df_produtos.count()} linhas)")

✓ dim_produtos (32951 linhas)


In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_sellers")

# Deduplicação: mantém apenas o registro mais recente por vendedor
window_dedup = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

df_vendedores = (df
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .withColumnRenamed("seller_id", "id_vendedor")
    .withColumnRenamed("seller_zip_code_prefix", "prefixo_cep")
    .withColumnRenamed("seller_city", "cidade")
    .withColumnRenamed("seller_state", "estado")
    .withColumnRenamed("seller_name", "nome_vendedor")
    .withColumnRenamed("seller_registration_date", "data_venda")
    # Padronização em maiúsculas para consistência em agrupamentos
    .withColumn("cidade", F.upper(F.col("cidade")))
    .withColumn("estado", F.upper(F.col("estado")))
    .drop("rank", "timestamp_ingestion")
)

# Apenas para validar
#df_vendedores.display()

(df_vendedores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_schema}.dim_vendedores"))

print(f"✓ dim_vendedores ({df_vendedores.count()} linhas)")

✓ dim_vendedores (3095 linhas)


In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_product_category_name_translation")

# Renomeia as colunas para o padrão da camada Silver
df_categorias = (df
    .withColumnRenamed("product_category_name",         "nome_produto_pt")
    .withColumnRenamed("product_category_name_english", "nome_produto_en")
    .drop("timestamp_ingestion")
)

# Apenas para validar
#df_categorias.display()

(df_categorias.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_schema}.dim_categoria_produtos_traducao"))

print(f"✓ dim_categoria_produtos_traducao ({df_categorias.count()} linhas)")

✓ dim_categoria_produtos_traducao (71 linhas)


In [0]:
df = spark.table(f"{catalogo}.{bronze_schema}.tb_cotacao_dolar")

# 🔥 Agora a data já existe e já está no tipo correto
df = df.withColumnRenamed("data_cotacao", "data")

# 🔥 Criar calendário DINÂMICO (melhor que fixo)
min_max = df.select(
    F.min("data").alias("min_data"),
    F.max("data").alias("max_data")
).collect()[0]

datas = (
    spark.sql(f"""
        SELECT sequence(
            to_date('{min_max['min_data']}'),
            to_date('{min_max['max_data']}'),
            interval 1 day
        ) as data
    """)
    .selectExpr("explode(data) as data")
)

# Join mantendo todos os dias
df_calendario = datas.join(
    df.select("data", "cotacaoCompra"),
    on="data",
    how="left"
)

# Window para preencher valores faltantes (fins de semana)
window_fill = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, 0)

df_cotacao = (
    df_calendario
    .withColumn(
        "cotacao_dolar",
        F.last("cotacaoCompra", ignorenulls=True).over(window_fill)
    )
    .withColumnRenamed("data", "data_cotacao")
    .drop("cotacaoCompra")
)

# Salvar tabela
(df_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_schema}.dim_cotacao_dolar")
)

print(f"✓ dim_cotacao_dolar ({df_cotacao.count()} linhas)")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ dim_cotacao_dolar (773 linhas)


In [0]:
from pyspark.sql import functions as F

df_pedidos    = spark.table(f"{catalogo}.{silver_schema}.fat_pedidos")
df_pagamentos = spark.table(f"{catalogo}.{silver_schema}.fat_pagamentos_pedidos")
df_cotacao    = spark.table(f"{catalogo}.{silver_schema}.dim_cotacao_dolar")

# Soma total sem arredondar ainda
df_pag_total = (
    df_pagamentos
    .groupBy("id_pedido")
    .agg(F.sum("valor_pagamento").alias("valor_total_pago_brl"))
)

df_pedido_total = (
    df_pedidos
    .withColumn("data_pedido", F.to_date("data_pedido"))
    
    .join(df_pag_total, on="id_pedido", how="left")
    
    # Join com cotação
    .join(
        df_cotacao,
        df_pedidos.data_pedido == df_cotacao.data_cotacao,
        how="left"
    )
    
    # Cálculo USD
    .withColumn(
        "valor_total_pago_usd",
        F.col("valor_total_pago_brl") / F.col("cotacao_dolar")
    )
    
    # 🔥 Arredondamento apenas no final (como exigido)
    .withColumn("valor_total_pago_brl", F.round("valor_total_pago_brl", 2))
    .withColumn("valor_total_pago_usd", F.round("valor_total_pago_usd", 2))
    
    .select(
        "id_pedido",
        "id_consumidor",
        "status",
        "valor_total_pago_brl",
        "valor_total_pago_usd",
        "data_pedido"
    )
)

#df_pedido_total.display()

(df_pedido_total.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_schema}.fat_pedido_total")
)

print(f"✓ fat_pedido_total ({df_pedido_total.count()} linhas)")

✓ fat_pedido_total (99441 linhas)


In [0]:
# Otimização física das tabelas fato para melhorar performance em queries analíticas
# ZORDER organiza os dados fisicamente pelas colunas mais usadas em filtros e joins
tabelas_fato = [
    ("fat_pedidos",            "id_pedido, data_pedido"),
    ("fat_itens_pedidos",      "id_pedido"),
    ("fat_pagamentos_pedidos", "id_pedido"),
    ("fat_pedido_total",       "id_pedido, data_pedido"),
]

for tabela, colunas in tabelas_fato:
    spark.sql(f"OPTIMIZE {catalogo}.{silver_schema}.{tabela} ZORDER BY ({colunas})")
    print(f"✓ OPTIMIZE concluído: {tabela}")


✓ OPTIMIZE concluído: fat_pedidos
✓ OPTIMIZE concluído: fat_itens_pedidos
✓ OPTIMIZE concluído: fat_pagamentos_pedidos
✓ OPTIMIZE concluído: fat_pedido_total
